# 02 - Camada Silver

A camada Silver é a **área de organização** do projeto.

Aqui, os dados que chegaram na Bronze são revisados, padronizados e preparados para uso.


In [0]:
#Configuração: definir as camadas de origem e destino

from pyspark.sql import functions as F

CATALOG = "nautical_lighthouse"
BRONZE_SCHEMA = "bronze"
SILVER_SCHEMA = "silver"


In [0]:
#Dados disponíveis

bronze_tables = (
    spark.sql(f"SHOW TABLES IN {CATALOG}.{BRONZE_SCHEMA}")
    .select("tableName")
)

display(bronze_tables)

In [0]:
#Diagnóstico inicial: É como revisar as caixas antes de reorganizar o estoque. Primeiro identificamos os problemas, depois decidimos o que realmente precisa ser corrigido.

quality_summary = []

for row in bronze_tables.collect():
    table_name = row["tableName"]

    df = spark.table(
        f"{CATALOG}.{BRONZE_SCHEMA}.{table_name}"
    )

    total_rows = df.count()
    total_columns = len(df.columns)

    duplicate_rows = (
        total_rows
        - df.drop(
            "source_file",
            "ingestion_timestamp"
        ).dropDuplicates().count()
    )

    quality_summary.append(
        (table_name, total_rows, total_columns, duplicate_rows)
    )

df_quality_summary = spark.createDataFrame(
    quality_summary,
    [
        "table_name",
        "row_count",
        "column_count",
        "duplicate_rows"
    ]
)

display(
    df_quality_summary.orderBy("table_name")
)

In [0]:
#Valores ausentes: contamos antes de decidir o tratamento deles

null_summary = []

for row in bronze_tables.collect():
    table_name = row["tableName"]

    df = spark.table(
        f"{CATALOG}.{BRONZE_SCHEMA}.{table_name}"
    )

    business_columns = [
        c for c in df.columns
        if c not in ["source_file", "ingestion_timestamp"]
    ]

    null_counts = df.select([
        F.sum(F.col(c).isNull().cast("int")).alias(c)
        for c in business_columns
    ]).collect()[0].asDict()

    for column_name, null_count in null_counts.items():
        if null_count > 0:
            null_summary.append(
                (table_name, column_name, null_count)
            )

df_null_summary = spark.createDataFrame(
    null_summary,
    ["table_name", "column_name", "null_count"]
)

display(
    df_null_summary.orderBy(
        F.desc("null_count")
    )
)

In [0]:
#Padronização básica sem alterar o significado dos dados

def basic_cleaning(df):
    for field in df.schema.fields:
        if field.dataType.simpleString() == "string":
            df = df.withColumn(
                field.name,
                F.trim(F.col(field.name))
            )

    return df

In [0]:
#Criação da camada Silver: uma tabela Silver por cada tabela Bronze

silver_results = []

for row in bronze_tables.collect():
    table_name = row["tableName"]

    source_table = (
        f"{CATALOG}.{BRONZE_SCHEMA}.{table_name}"
    )

    target_table = (
        f"{CATALOG}.{SILVER_SCHEMA}.{table_name}"
    )

    df = spark.table(source_table)

    df_clean = basic_cleaning(df)

    (
        df_clean.write
        .format("delta")
        .mode("overwrite")
        .saveAsTable(target_table)
    )

    silver_results.append(
        (table_name, df_clean.count())
    )

    print(
        f"{table_name}: {df_clean.count():,} registros"
    )

In [0]:
#Conferência de tabelas correspondentes

silver_tables = spark.sql(
    f"SHOW TABLES IN {CATALOG}.{SILVER_SCHEMA}"
)

display(silver_tables)

print(
    f"Tabelas Silver encontradas: "
    f"{silver_tables.count()}"
)

In [0]:
#Validação da carga
validation_results = []

for table_name, silver_count in silver_results:
    bronze_count = (
        spark.table(
            f"{CATALOG}.{BRONZE_SCHEMA}.{table_name}"
        ).count()
    )

    validation_results.append(
        (
            table_name,
            bronze_count,
            silver_count,
            bronze_count == silver_count
        )
    )

df_validation = spark.createDataFrame(
    validation_results,
    [
        "table_name",
        "bronze_rows",
        "silver_rows",
        "rows_match"
    ]
)

display(
    df_validation.orderBy("table_name")
)

In [0]:
#Investigar salesperson_id: pedidos sem vendedor

display(
    spark.table("nautical_lighthouse.bronze.orders")
    .groupBy("channel")
    .agg(
        F.count("*").alias("total_orders"),
        F.sum(
            F.col("salesperson_id").isNull().cast("int")
        ).alias("missing_salesperson")
    )
    .orderBy("channel")
)

In [0]:
#Investigar paid_at: Pagamentos sem data de pagamento

display(
    spark.table("nautical_lighthouse.bronze.payments")
    .groupBy("status")
    .agg(
        F.count("*").alias("total_payments"),
        F.sum(
            F.col("paid_at").isNull().cast("int")
        ).alias("missing_paid_at")
    )
    .orderBy("status")
)

In [0]:
#Investigar reorder_point: Ponto de reposição de estoque

stock_levels = spark.table(
    "nautical_lighthouse.bronze.stock_levels"
)

display(stock_levels.limit(20))

stock_levels.select(
    "reorder_point"
).distinct().show()

In [0]:
#Investigar movimentações sem funcionário

stock_movements = spark.table(
    "nautical_lighthouse.bronze.stock_movements"
)

print(stock_movements.columns)

In [0]:
#Conferir chaves primárias nulas

id_validation = []

for row in bronze_tables.collect():
    table_name = row["tableName"]

    df = spark.table(
        f"{CATALOG}.{BRONZE_SCHEMA}.{table_name}"
    )

    if "id" in df.columns:
        null_ids = df.filter(
            F.col("id").isNull()
        ).count()

        id_validation.append(
            (table_name, null_ids)
        )

df_id_validation = spark.createDataFrame(
    id_validation,
    ["table_name", "null_ids"]
)

display(df_id_validation.orderBy("table_name"))

In [0]:
#IDs duplicados
duplicate_id_validation = []

for row in bronze_tables.collect():
    table_name = row["tableName"]

    df = spark.table(
        f"{CATALOG}.{BRONZE_SCHEMA}.{table_name}"
    )

    if "id" in df.columns:
        duplicate_ids = (
            df.groupBy("id")
            .count()
            .filter(F.col("count") > 1)
            .count()
        )

        duplicate_id_validation.append(
            (table_name, duplicate_ids)
        )

df_duplicate_ids = spark.createDataFrame(
    duplicate_id_validation,
    ["table_name", "duplicate_ids"]
)

display(df_duplicate_ids.orderBy("table_name"))

### Decisões de qualidade

A análise mostrou que nem todo valor ausente representa um problema.

- **Pedidos sem vendedor:** ocorrem apenas no canal de e-commerce e foram preservados.
- **Pagamentos sem `paid_at`:** estão associados a pagamentos não aprovados e foram preservados.
- **Ponto de reposição:** `reorder_point` está vazio em toda a base. Como não há informação suficiente para estimá-lo com segurança, o campo foi mantido sem preenchimento.
- **Identificadores:** não foram encontrados IDs nulos ou duplicados.
- **Duplicidades:** não foram encontradas linhas completamente duplicadas.

Assim, evitamos criar informações artificiais apenas para eliminar valores nulos.

In [0]:
#Tabelas de relacionamento
composite_keys = {
    "product_suppliers": ["product_variant_id", "supplier_id"],
    "stock_levels": ["product_variant_id", "location_id"],
    "variant_attribute_values": ["product_variant_id", "attribute_id"]
}

composite_key_results = []

for table_name, keys in composite_keys.items():
    df = spark.table(
        f"{CATALOG}.{BRONZE_SCHEMA}.{table_name}"
    )

    duplicate_keys = (
        df.groupBy(*keys)
        .count()
        .filter(F.col("count") > 1)
        .count()
    )

    null_condition = None

    for key in keys:
        condition = F.col(key).isNull()
        null_condition = (
            condition
            if null_condition is None
            else null_condition | condition
        )

    null_keys = df.filter(null_condition).count()

    composite_key_results.append(
        (table_name, duplicate_keys, null_keys)
    )

df_composite_validation = spark.createDataFrame(
    composite_key_results,
    ["table_name", "duplicate_keys", "null_keys"]
)

display(df_composite_validation)

In [0]:
relationships = [
    ("orders", "customer_id", "customers", "id"),
    ("orders", "location_id", "locations", "id"),
    ("order_items", "order_id", "orders", "id"),
    ("order_items", "product_variant_id", "product_variants", "id"),
    ("product_variants", "product_id", "products", "id"),
    ("products", "brand_id", "brands", "id"),
    ("products", "category_id", "categories", "id"),
    ("payments", "order_id", "orders", "id"),
    ("fiscal_invoices", "order_id", "orders", "id"),
    ("returns", "order_id", "orders", "id"),
    ("return_items", "return_id", "returns", "id"),
    ("stock_levels", "product_variant_id", "product_variants", "id"),
    ("stock_levels", "location_id", "locations", "id")
]

relationship_results = []

for child_table, fk, parent_table, pk in relationships:

    child = spark.table(
        f"{CATALOG}.{BRONZE_SCHEMA}.{child_table}"
    )

    parent = spark.table(
        f"{CATALOG}.{BRONZE_SCHEMA}.{parent_table}"
    )

    orphan_count = (
        child
        .filter(F.col(fk).isNotNull())
        .join(
            parent.select(F.col(pk).alias(fk)),
            on=fk,
            how="left_anti"
        )
        .count()
    )

    relationship_results.append(
        (child_table, fk, parent_table, orphan_count)
    )

df_relationship_validation = spark.createDataFrame(
    relationship_results,
    [
        "table",
        "foreign_key",
        "reference_table",
        "orphan_records"
    ]
)

display(df_relationship_validation)

In [0]:
#Consistência financeira dos pedidos
orders = spark.table(
    f"{CATALOG}.{BRONZE_SCHEMA}.orders"
)

financial_check_orders = (
    orders
    .withColumn(
        "expected_total",
        F.round(F.col("subtotal") - F.col("discount_amount"), 2)
    )
    .withColumn(
        "difference",
        F.round(F.col("total") - F.col("expected_total"), 2)
    )
)

display(
    financial_check_orders
    .filter(F.abs(F.col("difference")) > 0.01)
    .select(
        "id",
        "subtotal",
        "discount_amount",
        "total",
        "expected_total",
        "difference"
    )
)
invalid_order_totals = (
    financial_check_orders
    .filter(F.abs(F.col("difference")) > 0.01)
    .count()
)

print(f"Pedidos com divergência financeira: {invalid_order_totals}")

In [0]:
#Consistência dos itens vendidos

order_items = spark.table(
    f"{CATALOG}.{BRONZE_SCHEMA}.order_items"
)

financial_check_items = (
    order_items
    .withColumn(
        "expected_line_total",
        F.round(
            F.col("quantity") * F.col("unit_price"),
            2
        )
    )
    .withColumn(
        "difference",
        F.round(
            F.col("line_total") - F.col("expected_line_total"),
            2
        )
    )
)

invalid_item_totals = (
    financial_check_items
    .filter(F.abs(F.col("difference")) > 0.01)
)

print(
    f"Itens com divergência financeira: "
    f"{invalid_item_totals.count():,}"
)

display(
    invalid_item_totals
    .select(
        "id",
        "order_id",
        "quantity",
        "unit_price",
        "line_total",
        "expected_line_total",
        "difference",
        "icms_rate",
        "ipi_rate"
    )
    .limit(20)
)

In [0]:
#Padronização dos valores categóricos

domain_checks = {
    "orders": ["channel", "status"],
    "payments": ["status"],
    "returns": ["status"],
    "stock_movements": ["movement_type"]
}

for table_name, columns in domain_checks.items():
    df = spark.table(
        f"{CATALOG}.{BRONZE_SCHEMA}.{table_name}"
    )

    print(f"\n--- {table_name} ---")

    for column in columns:
        if column in df.columns:
            print(f"\n{column}:")
            df.select(column).distinct().orderBy(column).show(truncate=False)

In [0]:
#Consistência entre as datas

temporal_results = []

for table_name in [
    "orders",
    "payments",
    "customers",
    "products",
    "purchase_orders",
    "returns"
]:
    df = spark.table(
        f"{CATALOG}.{BRONZE_SCHEMA}.{table_name}"
    )

    if "created_at" in df.columns and "updated_at" in df.columns:
        invalid_dates = (
            df.filter(
                F.col("updated_at") < F.col("created_at")
            ).count()
        )

        temporal_results.append(
            (table_name, invalid_dates)
        )

df_temporal_validation = spark.createDataFrame(
    temporal_results,
    ["table_name", "invalid_date_order"]
)

display(df_temporal_validation)

In [0]:
#Período dos pedidos

orders = spark.table(
    f"{CATALOG}.{BRONZE_SCHEMA}.orders"
)

display(
    orders.select(
        F.min("placed_at").alias("first_order"),
        F.max("placed_at").alias("last_order")
    )
)

In [0]:
#Conferência entre pedido e seus itens

orders = spark.table(
    f"{CATALOG}.{BRONZE_SCHEMA}.orders"
)

order_items = spark.table(
    f"{CATALOG}.{BRONZE_SCHEMA}.order_items"
)

items_by_order = (
    order_items
    .groupBy("order_id")
    .agg(
        F.round(
            F.sum("line_total"),
            2
        ).alias("items_total")
    )
)

order_reconciliation = (
    orders
    .select(
        F.col("id").alias("order_id"),
        "subtotal"
    )
    .join(
        items_by_order,
        on="order_id",
        how="left"
    )
    .withColumn(
        "difference",
        F.round(
            F.col("subtotal") - F.col("items_total"),
            2
        )
    )
)

invalid_reconciliation = (
    order_reconciliation
    .filter(
        F.col("items_total").isNull()
        | (F.abs(F.col("difference")) > 0.01)
    )
)

print(
    f"Pedidos com divergência entre itens e subtotal: "
    f"{invalid_reconciliation.count():,}"
)

display(
    invalid_reconciliation.limit(20)
)

In [0]:
#Conferência entre Bronze x Silver

final_validation = []

for row in bronze_tables.collect():
    table_name = row["tableName"]

    bronze_count = spark.table(
        f"{CATALOG}.{BRONZE_SCHEMA}.{table_name}"
    ).count()

    silver_count = spark.table(
        f"{CATALOG}.{SILVER_SCHEMA}.{table_name}"
    ).count()

    final_validation.append(
        (
            table_name,
            bronze_count,
            silver_count,
            bronze_count == silver_count
        )
    )

df_final_validation = spark.createDataFrame(
    final_validation,
    [
        "table_name",
        "bronze_rows",
        "silver_rows",
        "validated"
    ]
)

display(df_final_validation.orderBy("table_name"))

## Resultado da camada Silver

Antes de utilizar os dados nas análises, verificamos se eles estavam confiáveis.

As principais validações realizadas foram:

- ausência de registros duplicados
- identificadores válidos e únicos
- relações consistentes entre as tabelas
- valores ausentes avaliados de acordo com seu contexto
- cálculos de pedidos e itens financeiramente consistentes
- datas em ordem lógica
- período dos pedidos compatível com 2020–2026
- campos categóricos sem variações inesperadas

Valores ausentes que possuem uma explicação de negócio foram preservados, evitando criar informações artificiais apenas para preencher campos vazios.

Com essas validações, a camada Silver fornece uma base confiável para as análises da camada Gold.